In [1]:
!pip install editdistance rdflib pandas

In [2]:
import rdflib
from rdflib import Graph, URIRef, Literal
from rdflib import Namespace
from pathlib import Path
import os

In [3]:
CONFIG = {
    "Hosting": {
        "URL": "https://speakeasy.ifi.uzh.ch",
        "Username": "CyanPeekingMouse",
        "Password": "Qe5Hf3zJ"
    },
    "Data": {
        "Directory": "/space_mounts/atai-hs25/dataset",
        "local_dir": "./dataset",
        "NER": "./ner_dataset.csv",
        "Remote_Directory": "https://files.ifi.uzh.ch/ddis/teaching/2025/ATAI/dataset/",
    }
}

In [4]:
g = Graph()
data_dir = CONFIG["Data"]["Directory"]
local_dir = CONFIG["Data"]["local_dir"]
graph_path = os.path.join(data_dir, "graph.nt")
local_path = os.path.join(local_dir, "graph.nt")

if os.path.exists(graph_path):
    print(f"Loading graph at {graph_path}...")
    g.parse(graph_path, format="nt")
    print(f"Loaded {graph_path} → {len(g)} triples")
elif os.path.exists(local_path):
    print(f"Loading graph at {local_path}...")
    g.parse(local_path, format="nt")
    print(f"Loaded {local_path} → {len(g)} triples")
else:
    print(f"Graph does not exist at {graph_path} or {local_path}.")

Loading graph at ./dataset/graph.nt...
Loaded ./dataset/graph.nt → 2413825 triples


In [5]:
# get all subjects, predicates and objects from the graph
unique_subjects = set(g.subjects())
unique_predicates = set(g.predicates())
unique_objects = set(g.objects())

print(f"The graph has {len(unique_subjects)} unique subjects.")
print(f"The graph has {len(unique_predicates)} unique predicates.")
print(f"The graph has {len(unique_objects)} unique objects.")

print(f"predicates: {unique_predicates}")

The graph has 131078 unique subjects.
The graph has 502 unique predicates.
The graph has 466284 unique objects.
predicates: {rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1340'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P412'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P747'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P78'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P366'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P479'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1923'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P8839'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1445'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1809'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P200'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1366'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P3461'), rdflib.term.URIRef('http://www.wikida

In [6]:
def sample(values, n=10):
    values = list(values)
    return values[:n] if len(values) <= n else values[:n] + ["..."]

unique_subjects = list(set(g.subjects()))
unique_predicates = list(set(g.predicates()))
unique_objects = list(set(g.objects()))

print("Sample subjects:", sample(unique_subjects))
print("Sample predicates:", sample(unique_predicates))
print("Sample objects:", sample(unique_objects))

Sample subjects: [rdflib.term.URIRef('http://www.wikidata.org/entity/Q768946'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q2085004'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q471911'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q124505786'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q7037999'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q901825'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q4767588'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q691539'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q323201'), rdflib.term.URIRef('http://www.wikidata.org/entity/Q718537'), '...']
Sample predicates: [rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P1340'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P412'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P747'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P78'), rdflib.term.URIRef('http://www.wikidata.org/prop/direct/P366'), rdflib.te

# Graphical Model Tutorial - Named Entity Recognition

Ruijie Wang, Pascal Severin Andermatt | 12.10.2022  
Based on [Named Entity Recognition and Classification with Scikit-Learn](https://towardsdatascience.com/named-entity-recognition-and-classification-with-scikit-learn-f05372f07ba2), [ACNER](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus), [Hidden Markov Model for POS-tagging](https://medium.com/mlearning-ai/introduction-to-hidden-markov-model-hmm-with-simple-ner-d1353ff35842) and [sklearn-crfsuite](https://sklearn-crfsuite.readthedocs.io/en/latest/index.html).

## 1. Named Entity Recognition

* **Named Entity Recognition (NER)** is the process of recognizing information units like **names**, including person, organization and location names, and **numeric expressions** including time, date, money and percent expressions from **unstructured text**.

    * Who is the director of **Batman 1989**?


* NER is usually converted to the problem of **tagging** each word in the given text. 

    * For example, the tagging of the question **Who is the director of Batman 1989?** could be:
        
        | Id | Word | Tag | Description |
        | ---- | ---- | ---- | ---- |
        | 0 | Who | O | others |
        | 1 | is | O | - |
        | 2 | the | O | - |
        | 3 | director | O | - |
        | 4 | of | O | - |
        | 5 | Batman | **B-mov** | mov = movie |
        | 6 | 1989 | **I-mov** | - |
        | 7 | ? | O | - |

    * The above tages follow the [**Inside–outside–beginning (IOB) format**](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging)#cite_note-2): 
    
        * An O tag indicates that a token belongs to **no chunk**. In NER, it means the word is not an entity.
        
        * The B- prefix before a tag indicates that the tag is the **beginning** of a chunk.
        
        * The I- prefix before a tag indicates that the tag is **inside** a chunk.

## 2. Packages

* We will use **[pandas](https://pandas.pydata.org/)** and **[numpy](https://numpy.org/)** to read and manipulate the data.

In [7]:
!pip install pandas
!pip install numpy

* **[Scikit-learn](https://scikit-learn.org/stable/)** and **[sklearn-crfsuite](https://sklearn-crfsuite.readthedocs.io/en/latest/)** will be used to train baseline, HMM and CRF models.

In [8]:
!pip install -U 'scikit-learn<0.24'
!pip install sklearn-crfsuite

  Using cached scikit-learn-0.23.2.tar.gz (7.2 MB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for scikit-learn did not run successfully.
  │ exit code: 1
  ╰─> [63 lines of output]
      Ignoring numpy: markers 'python_version == "3.6" and platform_system != "AIX" and platform_python_implementation == "CPython"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.6" and platform_system != "AIX" and platform_python_implementation != "CPython"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.7" and platform_system != "AIX"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.6" and platform_system == "AIX"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.7" and platform_system == "AIX"' don't match your environment
      Ignoring numpy: markers 'python_version >= "3.8" and platform_system == "

- import required packages and modules

In [9]:
import pandas as pd
import numpy as np
import sklearn_crfsuite

from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics

from collections import Counter

## 3. Dataset

* **[Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)**

* An annotated corpus for Named Entity Recognition based on [GMB (Groningen Meaning Bank)](https://gmb.let.rug.nl/).

* There are mainly 8 types of tags:  

    | Tag | Description | Examples |
    | ---- | ---- | ---- |
    | geo | Geographical Entity | Paris, Washington DC |
    | org | Organization | APEC, WTO, United Nations |
    | per | Person | President George Bush |
    | gpe | Geopolitical Entity | U.S., France, Japan |
    | tim | Time indicator | July, Wednesday |
    | art | Artifact | Internet, International Space Station |
    | eve | Event | World War I, Tennis Masters Cup |
    | nat | Natural Phenomenon | Hurricane Katrina |

* There are two .csv files in the corpus:

    * **ner_dataset.csv** includes two features: `word` and `POS`. `POS` refers to part-of-speech tags. You can find a set of tags and descriptions at https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html.

    * **ner.csv** provides more features such as `shape`, `prev-word`, `next-word` that you can use to achieve better performance.

- Let's first use pandas to read the csv file. 

In [10]:
df = pd.read_csv('ner_dataset.csv', encoding = "ISO-8859-1")
df = df[:10000]  # only take the first 10,000 records for this tutorial
df = df.fillna(method='ffill')
df[:5]

/var/folders/0j/5j76lpw55lb8_mzj13c6_ls00000gn/T/ipykernel_4644/3797048013.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O


In [11]:
print("- number of sentences: {},\n- number of unique words: {},\n- number of unique POS tags: {},\n- number of unique NER tags: {}".format(df['Sentence #'].nunique(), df.Word.nunique(), df.POS.nunique(), df.Tag.nunique()))

- number of sentences: 457,
- number of unique words: 2746,
- number of unique POS tags: 39,
- number of unique NER tags: 17


In [12]:
# show the distribution of NER tags
df.groupby('Tag').size().reset_index(name='Counts')

,Tag,Counts
0,B-art,28
1,B-eve,10
2,B-geo,244
3,B-gpe,303
4,B-nat,5
5,B-org,176
6,B-per,160
7,B-tim,149
8,I-art,20
9,I-eve,10


## 4. A Linear classifier with SGD training

- A **support vector machine (SVM)** learns hyperplanes to seperate words with different tags.

- Each word is classified independently by the classifier based on its features.

- Dependencies between words are not considered in this tutorial.

In [13]:
# only leave Word and POS as input features
X = df.drop('Tag', axis=1).drop('Sentence #', axis=1)
X

,Word,POS
0,Thousands,NNS
1,of,IN
2,demonstrators,NNS
3,have,VBP
4,marched,VBN
...,...,...
9995,war,NN
9996,crimes,NNS
9997,tribunal,NN
9998,in,IN


In [14]:
# vectorize X
v = DictVectorizer(sparse=False)
X = v.fit_transform(X.to_dict('records'))
print("--- the shape of X after vectorization: {}".format(X.shape))
X

--- the shape of X after vectorization: (10000, 2785)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(10000, 2785))

In [15]:
# NER targets
y = df.Tag.values
print("--- the shape of y: {}".format(X.shape))
y[:22]

--- the shape of y: (10000, 2785)


array(['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O',
       'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O'],
      dtype=object)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state=0)

print("--- number of training and test words: {}, {}".format( X_train.shape[0], X_test.shape[0]))
print("--- dimension of input features: {}".format(X_train.shape[1]))

--- number of training and test words: 6700, 3300
--- dimension of input features: 2785


In [17]:
# leave all other parameters to default
sgd = SGDClassifier(loss="hinge", max_iter=1000)  # https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html

# train the model
sgd.fit(X_train, y_train)

classifier_res = classification_report(y_pred=sgd.predict(X_test), y_true=y_test)
print(classifier_res)

              precision    recall  f1-score   support

       B-art       1.00      0.22      0.36         9
       B-eve       0.50      0.33      0.40         3
       B-geo       0.82      0.48      0.61        69
       B-gpe       0.76      0.68      0.72       102
       B-nat       0.00      0.00      0.00         0
       B-org       0.82      0.51      0.63        63
       B-per       0.81      0.54      0.65        41
       B-tim       0.96      0.87      0.91        52
       I-art       0.00      0.00      0.00        10
       I-eve       1.00      0.33      0.50         3
       I-geo       0.75      0.27      0.40        11
       I-gpe       0.67      0.33      0.44         6
       I-nat       0.00      0.00      0.00         1
       I-org       0.72      0.45      0.55        47
       I-per       0.36      0.92      0.52        66
       I-tim       0.17      0.25      0.20         4
           O       0.99      1.00      0.99      2813

    accuracy              

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

* F1-score is computed based on precision and recall: $2*((precision*recall)/(precision+recall))$
* Macro Average Precision is the average precision w.r.t all classes.
* Weighted Average Precision is the weighted average precision of all classes.

## HMM-based POS tagging

* Tags and words are respectively considered as hidden states and observations.

* Two matrices would be needed for HMM modeling:

    * transition probability matrix
    
    * observation probability matrix

In [18]:
def collate(dataframe):
    agg_func = lambda s: [(w, t) for w, t in zip(s['Word'].values.tolist(), s['Tag'].values.tolist())]
    grouped = dataframe.groupby('Sentence #').apply(agg_func)
    return list(grouped)

print("--- original dataframe:\n")
print(df)
sentences = collate(df)
print("\n--- number of sentences: {}\n".format(len(sentences)))
sentences[0:2]

--- original dataframe:

         Sentence #           Word  POS Tag
0       Sentence: 1      Thousands  NNS   O
1       Sentence: 1             of   IN   O
2       Sentence: 1  demonstrators  NNS   O
3       Sentence: 1           have  VBP   O
4       Sentence: 1        marched  VBN   O
...             ...            ...  ...  ..
9995  Sentence: 457            war   NN   O
9996  Sentence: 457         crimes  NNS   O
9997  Sentence: 457       tribunal   NN   O
9998  Sentence: 457             in   IN   O
9999  Sentence: 457            The   DT   O

[10000 rows x 4 columns]

--- number of sentences: 457



/var/folders/0j/5j76lpw55lb8_mzj13c6_ls00000gn/T/ipykernel_4644/1555979887.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = dataframe.groupby('Sentence #').apply(agg_func)


[[('Thousands', 'O'),
  ('of', 'O'),
  ('demonstrators', 'O'),
  ('have', 'O'),
  ('marched', 'O'),
  ('through', 'O'),
  ('London', 'B-geo'),
  ('to', 'O'),
  ('protest', 'O'),
  ('the', 'O'),
  ('war', 'O'),
  ('in', 'O'),
  ('Iraq', 'B-geo'),
  ('and', 'O'),
  ('demand', 'O'),
  ('the', 'O'),
  ('withdrawal', 'O'),
  ('of', 'O'),
  ('British', 'B-gpe'),
  ('troops', 'O'),
  ('from', 'O'),
  ('that', 'O'),
  ('country', 'O'),
  ('.', 'O')],
 [('Iranian', 'B-gpe'),
  ('officials', 'O'),
  ('say', 'O'),
  ('they', 'O'),
  ('expect', 'O'),
  ('to', 'O'),
  ('get', 'O'),
  ('access', 'O'),
  ('to', 'O'),
  ('sealed', 'O'),
  ('sensitive', 'O'),
  ('parts', 'O'),
  ('of', 'O'),
  ('the', 'O'),
  ('plant', 'O'),
  ('Wednesday', 'B-tim'),
  (',', 'O'),
  ('after', 'O'),
  ('an', 'O'),
  ('IAEA', 'B-org'),
  ('surveillance', 'O'),
  ('system', 'O'),
  ('begins', 'O'),
  ('functioning', 'O'),
  ('.', 'O')]]

In [19]:
train_sentences, test_sentences = train_test_split(sentences, test_size=0.3, random_state=0)
print('- number of training sentences: ', len(train_sentences))
print('- number of testing sentences: ', len(test_sentences))

- number of training sentences:  319
- number of testing sentences:  138


In [20]:
# count of word-tag occurrences
token_list = [word for sentence in train_sentences for word in sentence]
tokens, tags = list(zip(*token_list))
print("--- number of training tokens: {}".format(len(tokens)))


hmm_df = pd.DataFrame(tokens, tags).reset_index().rename(columns={0: 'tokens', 'index': 'tag'})
hmm_df['values'] = 1
hmm_df

--- number of training tokens: 6943


,tag,tokens,values
0,B-per,Mr.,1
1,I-per,Sivaram,1
2,O,",",1
3,O,who,1
4,O,was,1
...,...,...,...
6938,O,contact,1
6939,O,with,1
6940,O,dead,1
6941,O,birds,1


In [21]:
# compute the observation probability matrix
hmm_df = hmm_df.pivot_table(index='tag', columns='tokens', aggfunc='sum').fillna(0)

sum_ = hmm_df.sum(axis=1)
hmm_df = hmm_df.div(sum_, axis=0)
hmm_df.columns = hmm_df.columns.droplevel(0)

hmm_df

hmm_df.style.highlight_max(color = 'lightgreen', axis = 1)

In [22]:
# compute transition probability matrix
df2 = []
for sentence in train_sentences:
    for index in range(len(sentence) - 1):
        first_token = sentence[index][1]
        next_token = sentence[index + 1][1]
        df2.append({'first_token': first_token, 'next_token': next_token})

df2 = pd.DataFrame(df2)

df2['values'] = 1
df2 = df2.pivot_table(index='first_token', columns='next_token', aggfunc='sum').fillna(0)
df2 = df2.div(df2.sum(axis=1), axis=0)
df2.columns = df2.columns.droplevel(0)
df_colums = df2.columns.tolist()

# compute probability distribution of the first token
start_tag = '<s>'
prob_start = {'B': 0, 'I': 0, 'O': 0}
for sentence in train_sentences:
    # summarize IOB tags to only keep the first letter (eg. B-art -> B)
    tag = sentence[0][1]
    prob_start[tag[0]] += 1
prob_start = {k: v / sum(prob_start.values()) for k, v in prob_start.items()}
print("Probabilities that the first token is a B, I, or O: {}".format(prob_start))
probs_data = [prob_start[i[0]] for i in df_colums]

df2 = pd.concat([pd.DataFrame(columns=df_colums, index=[start_tag], data=[probs_data]), df2], axis=0)

df2.style.highlight_max(color = 'lightgreen', axis = 1)

Probabilities that the first token is a B, I, or O: {'B': 0.29780564263322884, 'I': 0.0, 'O': 0.7021943573667712}


,B-art,B-eve,B-geo,B-gpe,B-nat,B-org,B-per,B-tim,I-art,I-eve,I-geo,I-gpe,I-org,I-per,I-tim,O
,0.297806,0.297806,0.297806,0.297806,0.297806,0.297806,0.297806,0.297806,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.702194
B-art,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.391304,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.608696
B-eve,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.875000,0.000000,0.000000,0.000000,0.000000,0.000000,0.125000
B-geo,0.000000,0.000000,0.000000,0.005882,0.000000,0.000000,0.000000,0.035294,0.000000,0.000000,0.076471,0.000000,0.000000,0.000000,0.000000,0.882353
B-gpe,0.000000,0.000000,0.000000,0.000000,0.000000,0.026549,0.088496,0.000000,0.000000,0.000000,0.000000,0.061947,0.000000,0.000000,0.000000,0.823009
B-nat,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
B-org,0.007874,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.440945,0.000000,0.000000,0.551181
B-per,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.801887,0.000000,0.198113
B-tim,0.000000,0.000000,0.009091,0.009091,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.081818,0.900000
I-art,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.437500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.562500


In [23]:
def predict(sentence):
    tokens = [token for (token, _) in sentence]
    tags = [] # empty because we will fill it with the predicted tags
    previous_tag = start_tag

    for token in tokens:
        if token not in hmm_df.columns:
            previous_tag = 'O'
        else:
            matrix = hmm_df[token].multiply(df2.loc[previous_tag])
            previous_tag = hmm_df.index[np.argmax(matrix)]
        token_tag_pair = (token, previous_tag)
        tags.append(token_tag_pair)
    
    return tags

sentence = test_sentences[1]
print(f"original sentence: {sentence}\n")
print(f"predicted tags: {predict(sentence)}")


original sentence: [('Kony', 'B-per'), ('is', 'O'), ('one', 'O'), ('of', 'O'), ('five', 'O'), ('LRA', 'B-org'), ('officials', 'O'), ('sought', 'O'), ('by', 'O'), ('the', 'O'), ('International', 'B-org'), ('Criminal', 'I-org'), ('Court', 'I-org'), ('for', 'O'), ('alleged', 'O'), ('war', 'O'), ('crimes', 'O'), ('.', 'O')]

predicted tags: [('Kony', 'B-art'), ('is', 'O'), ('one', 'O'), ('of', 'O'), ('five', 'O'), ('LRA', 'B-org'), ('officials', 'O'), ('sought', 'O'), ('by', 'O'), ('the', 'O'), ('International', 'B-org'), ('Criminal', 'O'), ('Court', 'B-art'), ('for', 'O'), ('alleged', 'O'), ('war', 'O'), ('crimes', 'O'), ('.', 'O')]


In [24]:
import matplotlib.pyplot as plt
import seaborn as sn
def evaluate(test_sentences):
    y_true = []
    y_pred = []
    for sentence in test_sentences:
        y_true.extend([tag for (_, tag) in sentence])
        y_pred.extend([tag for (_, tag) in predict(sentence)])

    return classification_report(y_true, y_pred)

hmm_res = evaluate(test_sentences)
print("--- performance of the HMM model")
print(hmm_res)

--- performance of the HMM model
              precision    recall  f1-score   support

       B-art       0.03      0.40      0.05         5
       B-eve       0.50      0.50      0.50         2
       B-geo       0.68      0.44      0.53        73
       B-gpe       0.78      0.66      0.72        77
       B-nat       0.00      0.00      0.00         2
       B-org       0.71      0.41      0.52        49
       B-per       0.79      0.61      0.69        54
       B-tim       0.93      0.95      0.94        39
       I-art       0.00      0.00      0.00         4
       I-eve       0.00      0.00      0.00         1
       I-geo       1.00      0.06      0.12        16
       I-gpe       1.00      1.00      1.00         3
       I-nat       0.00      0.00      0.00         2
       I-org       1.00      0.19      0.32        48
       I-per       0.97      0.43      0.60        65
       I-tim       0.00      0.00      0.00         1
           O       0.95      0.99      0.97     

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

## Conditional Random Fields (CRFs)

* The above HMM model only considers transition between hidden states (tags) and transition between hidden states (tags) and words.

* There are actually many other features that we can use, e.g., POS tags of words and dependences between words.

* How can we improve the NER performance by considering more features?

In [25]:
def collate(dataframe):
    agg_func = lambda s: [(w, pos, t) for w, pos, t in zip(s['Word'].values.tolist(), s['POS'].values.tolist(), s['Tag'].values.tolist())]
    grouped = dataframe.groupby('Sentence #').apply(agg_func)
    return list(grouped)

print("--- original dataframe:\n")
print(df)
sentences = collate(df)
print("\n--- number of sentences: {}\n".format(len(sentences)))
sentences[0:2]

--- original dataframe:

         Sentence #           Word  POS Tag
0       Sentence: 1      Thousands  NNS   O
1       Sentence: 1             of   IN   O
2       Sentence: 1  demonstrators  NNS   O
3       Sentence: 1           have  VBP   O
4       Sentence: 1        marched  VBN   O
...             ...            ...  ...  ..
9995  Sentence: 457            war   NN   O
9996  Sentence: 457         crimes  NNS   O
9997  Sentence: 457       tribunal   NN   O
9998  Sentence: 457             in   IN   O
9999  Sentence: 457            The   DT   O

[10000 rows x 4 columns]

--- number of sentences: 457



/var/folders/0j/5j76lpw55lb8_mzj13c6_ls00000gn/T/ipykernel_4644/2027667274.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = dataframe.groupby('Sentence #').apply(agg_func)


[[('Thousands', 'NNS', 'O'),
  ('of', 'IN', 'O'),
  ('demonstrators', 'NNS', 'O'),
  ('have', 'VBP', 'O'),
  ('marched', 'VBN', 'O'),
  ('through', 'IN', 'O'),
  ('London', 'NNP', 'B-geo'),
  ('to', 'TO', 'O'),
  ('protest', 'VB', 'O'),
  ('the', 'DT', 'O'),
  ('war', 'NN', 'O'),
  ('in', 'IN', 'O'),
  ('Iraq', 'NNP', 'B-geo'),
  ('and', 'CC', 'O'),
  ('demand', 'VB', 'O'),
  ('the', 'DT', 'O'),
  ('withdrawal', 'NN', 'O'),
  ('of', 'IN', 'O'),
  ('British', 'JJ', 'B-gpe'),
  ('troops', 'NNS', 'O'),
  ('from', 'IN', 'O'),
  ('that', 'DT', 'O'),
  ('country', 'NN', 'O'),
  ('.', '.', 'O')],
 [('Iranian', 'JJ', 'B-gpe'),
  ('officials', 'NNS', 'O'),
  ('say', 'VBP', 'O'),
  ('they', 'PRP', 'O'),
  ('expect', 'VBP', 'O'),
  ('to', 'TO', 'O'),
  ('get', 'VB', 'O'),
  ('access', 'NN', 'O'),
  ('to', 'TO', 'O'),
  ('sealed', 'JJ', 'O'),
  ('sensitive', 'JJ', 'O'),
  ('parts', 'NNS', 'O'),
  ('of', 'IN', 'O'),
  ('the', 'DT', 'O'),
  ('plant', 'NN', 'O'),
  ('Wednesday', 'NNP', 'B-tim'),
  ('

In [26]:
train_sentences, test_sentences = train_test_split(sentences, test_size=0.3, random_state=0)
print('- number of training sentences: ', len(train_sentences))
print('- number of testing sentences: ', len(test_sentences))
print(train_sentences[0])

- number of training sentences:  319
- number of testing sentences:  138
[('Mr.', 'NNP', 'B-per'), ('Sivaram', 'NNP', 'I-per'), (',', ',', 'O'), ('who', 'WP', 'O'), ('was', 'VBD', 'O'), ('also', 'RB', 'O'), ('brutally', 'RB', 'O'), ('attacked', 'VBN', 'O'), ('in', 'IN', 'O'), ('2001', 'CD', 'B-tim'), (',', ',', 'O'), ('was', 'VBD', 'O'), ('found', 'VBN', 'O'), ('near', 'IN', 'O'), ('a', 'DT', 'O'), ('lake', 'NN', 'O'), ('gagged', 'VBN', 'O'), ('with', 'IN', 'O'), ('gunshot', 'NN', 'O'), ('wounds', 'NNS', 'O'), ('to', 'TO', 'O'), ('the', 'DT', 'O'), ('head', 'NN', 'O'), ('.', '.', 'O')]


* We consider the following features in the CRF:

    * POS tags
    * Word parts
    * Lower/title/upper flags
    * IBO prefixes
    * Features of neighboring words


* As required by sklearn-crfsuite, each sentence should be converted to a list of dicts.

In [27]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]
    
    features = {
        'word.lower()': word.lower(),  # the word in lowercase
        'word[-3:]': word[-3:],  # last three characters
        'word[-2:]': word[-2:],  # last two characters
        'word.isupper()': word.isupper(),  # true, if the word is in uppercase
        'word.istitle()': word.istitle(),  # true, if the first character is in uppercase and remaining characters are in lowercase
        'word.isdigit()': word.isdigit(),  # true, if all characters are digits
        'postag': postag,  # POS tag
        'postag[:2]': postag[:2],  # IOB prefix
    }
    
    if i > 0:
        word1 = sent[i-1][0]  # the previous word
        postag1 = sent[i-1][1]  # POS tag of the previous word
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })  # add some features of the previous word
    else:
        features['BOS'] = True  # BOS: begining of the sentence
        
    if i < len(sent)-1:
        word1 = sent[i+1][0]  # the next word
        postag1 = sent[i+1][1]  # POS tag of the next word
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })  # add some features of the next word
    else:
        features['EOS'] = True  # EOS: end of the sentence
    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for _, _, label in sent]

In [28]:
# example
print("--- extracted features of the first word of the first training sentence:\n")
for key, value in sent2features(train_sentences[0])[0].items():
    print(key, value)

print("\n--- extracted features of the second word of the first training sentence:\n")
for key, value in sent2features(train_sentences[0])[1].items():
    print(key, value)
    
print("\n--- NER tag of the first word:\n")
print(sent2labels(train_sentences[0])[0])

--- extracted features of the first word of the first training sentence:

word.lower() mr.
word[-3:] Mr.
word[-2:] r.
word.isupper() False
word.istitle() True
word.isdigit() False
postag NNP
postag[:2] NN
BOS True
+1:word.lower() sivaram
+1:word.istitle() True
+1:word.isupper() False
+1:postag NNP
+1:postag[:2] NN

--- extracted features of the second word of the first training sentence:

word.lower() sivaram
word[-3:] ram
word[-2:] am
word.isupper() False
word.istitle() True
word.isdigit() False
postag NNP
postag[:2] NN
-1:word.lower() mr.
-1:word.istitle() True
-1:word.isupper() False
-1:postag NNP
-1:postag[:2] NN
+1:word.lower() ,
+1:word.istitle() False
+1:word.isupper() False
+1:postag ,
+1:postag[:2] ,

--- NER tag of the first word:

B-per


In [29]:
# X and y in the required format
X_train, y_train = [sent2features(s) for s in train_sentences], [sent2labels(s) for s in train_sentences]
X_test, y_test = [sent2features(s) for s in test_sentences], [sent2labels(s) for s in test_sentences]

- Train a CRF model

In [30]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',    # Limited-memory BFGS generally performs better than SGD
    c1=0.1,              # L1 regularization to encourage sparsity
    c2=0.1,              # L2 regularization to prevent overfitting
    max_iterations=1000,  # Maximum iterations
    all_possible_transitions=True,  # Include all possible state transitions
    all_possible_states=True       # Include all possible states
)

In [31]:
crf.fit(X_train, y_train)

,algorithm,'lbfgs'
,min_freq,None
,all_possible_states,True
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,1000
,num_memories,None
,epsilon,None
,period,None
,delta,None


In [32]:
y_pred = crf.predict(X_test)

print("--- performance of the CRF model")
print(metrics.flat_classification_report(y_test, y_pred))

print("--- performance of the HMM model")
print(hmm_res)

--- performance of the CRF model
              precision    recall  f1-score   support

       B-art       0.67      0.40      0.50         5
       B-eve       1.00      0.50      0.67         2
       B-geo       0.88      0.73      0.80        73
       B-gpe       0.74      0.90      0.81        77
       B-nat       0.00      0.00      0.00         2
       B-org       0.77      0.67      0.72        49
       B-per       0.80      0.91      0.85        54
       B-tim       0.92      0.87      0.89        39
       I-art       0.00      0.00      0.00         4
       I-eve       0.00      0.00      0.00         1
       I-geo       1.00      0.44      0.61        16
       I-gpe       0.60      1.00      0.75         3
       I-nat       0.00      0.00      0.00         2
       I-org       0.84      0.67      0.74        48
       I-per       0.80      0.98      0.88        65
       I-tim       0.33      1.00      0.50         1
           O       0.99      1.00      1.00     

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

- The performance has been significantly improved by considering more features of given words and dependencies between neighboring words.

## Discussion - What has the CRF model learned?

- Check the learned dependencies between words:

In [33]:
def print_transitions(trans_features):
    for (label_from, label_to), weight in trans_features:
        print("%-6s -> %-7s %0.6f" % (label_from, label_to, weight))
        
print("Top likely transitions:")
print_transitions(Counter(crf.transition_features_).most_common(20))

print("\nTop unlikely transitions:")
print_transitions(Counter(crf.transition_features_).most_common()[-20:])

Top likely transitions:
O      -> O       5.328540
B-org  -> I-org   4.428851
B-per  -> I-per   4.355233
I-org  -> I-org   3.873345
I-per  -> I-per   3.841282
B-eve  -> I-eve   3.643716
B-art  -> I-art   3.556894
I-art  -> I-art   3.365270
B-gpe  -> I-gpe   3.179459
B-geo  -> I-geo   2.869089
I-gpe  -> I-gpe   2.478205
B-tim  -> I-tim   2.264447
O      -> B-tim   2.221054
O      -> B-gpe   2.165559
O      -> B-org   1.896838
I-tim  -> I-tim   1.839572
O      -> B-geo   1.819949
I-eve  -> I-eve   1.808942
I-geo  -> I-geo   1.710072
O      -> B-art   1.540056

Top unlikely transitions:
O      -> I-eve   -0.452346
B-geo  -> I-gpe   -0.458247
I-org  -> B-org   -0.462176
B-gpe  -> I-art   -0.505097
B-eve  -> O       -0.548427
O      -> I-geo   -0.575289
B-geo  -> I-art   -0.670749
B-gpe  -> I-org   -0.680458
B-tim  -> B-gpe   -0.682593
B-org  -> I-per   -0.686014
O      -> I-gpe   -0.723177
B-geo  -> I-org   -0.739040
B-geo  -> I-per   -0.839693
I-org  -> I-per   -0.880467
B-gpe  -> I-per  

- Check how the CRF considers given features of words:

In [34]:
def print_state_features(state_features):
    for (attr, label), weight in state_features:
        print("%-6s -> %-7s %0.6f" % (attr, label, weight))
        
print("Top positive:")
print_state_features(Counter(crf.state_features_).most_common(30))

print("\nTop negative:")
print_state_features(Counter(crf.state_features_).most_common()[-30:])

Top positive:
BOS    -> O       5.226602
word[-3:]:day -> B-tim   3.613666
word[-2:]:ay -> B-tim   3.542119
word.istitle() -> B-gpe   2.958848
postag:NN -> O       2.662848
word.isupper() -> B-org   2.447881
EOS    -> O       2.316570
-1:word.lower():in -> B-geo   2.290057
postag:JJ -> B-gpe   2.065211
word[-2:]:0s -> B-tim   2.046434
-1:word.lower():in -> B-tim   2.024097
postag:NNS -> O       1.937708
word.isdigit() -> B-tim   1.918878
word[-3:]:ber -> B-tim   1.899161
word[-3:]:ban -> B-org   1.851275
word[-2:]:na -> B-gpe   1.804889
word.isdigit() -> I-tim   1.786500
-1:postag[:2]:NN -> O       1.777563
-1:word.lower():with -> B-gpe   1.759508
-1:word.lower():from -> B-geo   1.746007
postag[:2]:VB -> O       1.744182
word[-2:]:ic -> O       1.734552
word[-2:]:am -> B-per   1.653411
-1:word.lower():u.s. -> B-org   1.648596
word[-3:]:sia -> B-geo   1.640629
word[-3:]:dan -> B-gpe   1.621076
word.lower():u.s. -> B-gpe   1.607474
word[-3:]:.S. -> B-gpe   1.607474
word[-2:]:S. -> B-gpe 

## NER for Question Answering

- NER could be very useful for your final project because recognizing named entities in given natural language questions is usually the very first step. 

- The user poses a question **"who is the director of Batman 1989"**.

In [35]:
questions = {
    "Who is the director of Good Will Hunting?": "Gus Van Sant is the director of Good Will Hunting.",
    "Who directed The Bridge on the River Kwai?": "David Lean directed The Bridge on the River Kwai.",
    "Who is the director of Star Wars: Episode VI - Return of the Jedi": "David Lean directed The Bridge on the River Kwai.",
    "Who is the screenwriter of The Masked Gang: Cyprus?": "The answer suggested by embeddings: Cengiz Küçükayvaz, Murat Aslan, and Melih Ekener.",
    "What is the MPAA film rating of Weathering with You?": "According to embeddings, the MPAA film rating of Weathering with You is PG-13.",
    "What is the genre of Good Neighbors?": "The genre of Good Neighbors is likely to be drama, comedy-drama, and comedy film.",
    "Who is the director of Batman 1989?": "Hi, the director of Batman 1989 is Timothy Walter Burton",
}

- **"Batman 1989"** can be recognized as an entity via NER.

In [36]:
# Implement NER method to extract the entity from the question
def extract_entity(question):
    # Convert question into the format expected by the CRF model
    question = question[0].lower() + question[1:] # make first letter of question lowercase
    # remove any marking punctuation
    question = question.replace('?', '').replace('.', '').replace('!', '')

    words = question.split() # split into words
    
    # Assign more specific POS tags based on capitalization and position
    pos_tags = []
    for i, word in enumerate(words):
        if word.istitle() or word.isupper() or word.isdigit():
            pos_tags.append('NNP')  # Proper noun
        elif word.lower() in ['who', 'what', 'where', 'when', 'why', 'how']:
            pos_tags.append('WP')  # Wh-pronoun
        elif word.lower() in ['is', 'are', 'was', 'were']:
            pos_tags.append('VBZ')  # Verb
        elif word.lower() in ['the', 'a', 'an']:
            pos_tags.append('DT')  # Determiner
        elif word.lower() in ['of', 'in', 'by', 'with']:
            pos_tags.append('IN')  # Preposition
        else:
            pos_tags.append('NN')  # Common noun
    
    # Create sentence structure with dummy third element to match training data format
    sentence = [(word, pos, 'O') for word, pos in zip(words, pos_tags)]
    
    # Extract features
    X = [word2features(sent=sentence, i=i) for i in range(len(sentence))]
    
    # Predict tags
    tags = crf.predict([X])[0]
    
    # Extract entity with more sophisticated logic
    entity = []
    in_entity = False
    
    for word, tag in zip(words, tags):
        if tag.startswith('B-'):  # Beginning of entity
            if in_entity:  # If we were already in an entity, store it and start new one
                entity.append(' ')
            entity.append(word)
            in_entity = True
        elif tag.startswith('I-') and in_entity:  # Inside of entity
            entity.append(' ' + word)
        elif not tag.startswith('I-'):  # Outside of entity
            in_entity = False
    
    result = ''.join(entity) if entity else None
    
    # Debug information
    #print("Words:", words)
    #print("POS Tags:", pos_tags)
    #print("NER Tags:", tags)
    
    return result

In [37]:
for question in questions.keys():
    print(f"Question: {question}")
    # Test the NER method on our question
    entity = extract_entity(question)
    print(f"Extracted entity: {entity}")

Question: Who is the director of Good Will Hunting?
Extracted entity: Good Will Hunting
Question: Who directed The Bridge on the River Kwai?
Extracted entity: The BridgeRiver Kwai
Question: Who is the director of Star Wars: Episode VI - Return of the Jedi
Extracted entity: Star Wars: Episode VIReturnJedi
Question: Who is the screenwriter of The Masked Gang: Cyprus?
Extracted entity: The Masked Gang: Cyprus
Question: What is the MPAA film rating of Weathering with You?
Extracted entity: MPAAWeatheringYou
Question: What is the genre of Good Neighbors?
Extracted entity: Good Neighbors
Question: Who is the director of Batman 1989?
Extracted entity: Batman 1989


- By defining a **question pattern**, the **relation associated with the entity** can be recognized;

In [38]:
import re

# a naive way for matching entities and relations

question_pattern = "who is the (.*) of ENTITY"

print("question pattern: {}\n".format(question_pattern))

question = re.sub(entity, "ENTITY", question.rstrip("?"))  # preprocess the question

relation = re.match(question_pattern, question).group(1)  # match the relation using a pattern

print("recognized relation: {}\n".format(relation))

question pattern: who is the (.*) of ENTITY



AttributeError: 'NoneType' object has no attribute 'group'

- Given "Batman 1989" and "director", find the corresponding entity and predicate in the knowledge graph: `<Batman_1989>` and `<director>`

In [ ]:
nodes = {}
predicates = {}

for node in g.all_nodes():
    if isinstance(node, URIRef):
        if g.value(node, n.label):
            nodes[node.toPython()] = g.value(node, n.label).toPython()
        else:
            nodes[node.toPython()] = re.sub("http://example.org/", "", node.toPython())

for s, p, o in g:
    predicates[p.toPython()] = re.sub("http://example.org/", "", p.toPython())

print("labeled nodes: {}\n".format(nodes))
print("predicates: {}\n".format(predicates))

In [ ]:
import json
import editdistance

def link_label(surface: str) -> tuple[str | None, float]:
    best_distance = 9999
    best_URI = ""
    if not surface:
        return (None, best_distance)

    with open("labels/inverted_index.json", "r", encoding="utf-8") as f:
        inverted_index = json.load(f)

    for key, value in inverted_index.items():
        tmp_distance = editdistance.eval(key, surface)
        if tmp_distance < best_distance:
            best_label, best_distance, best_URI = key, tmp_distance, value
    return (best_label, best_URI, best_distance)

In [ ]:
print(link_label("Star Wars: Episode VIReturnJedi"))

In [ ]:
for question in questions.keys():
    print(f"Question: {question}")
    # Test the NER method on our question
    entity = extract_entity(question)
    print(link_label(entity))

- The corresponding SPARQL query can be constructed according to a SPARQL template: `SELECT ?x WHERE {?x, PREDICATE, NODE.}`

In [ ]:
query_template = "SELECT DISTINCT ?x ?y WHERE {{ ?x <{}> <{}>. ?x <{}> ?y. }}".format(match_pred, match_node, n.label)

print("--- sparql query: {}".format(query_template))

qres = g.query(query_template)

print("\n--- querying results: ")
for row in qres:
    print(row.x, row.y)
    answer = row.y

- Finally, we can define response templates to generate natural language responses:

In [ ]:
answer_template = "Hi, the {} of {} is {}".format(relation, entity, answer)

print("\n--- generated response: {}".format(answer_template))